# Notebook 24 — Continuous Universality Density Fields

This notebook turns the residual manifold from previous notebooks into a **continuous soft-assignment field**.

Instead of assigning each controlled/OOD graph to one nearest known family, we estimate a smooth density for each known topology in residual-manifold coordinates and evaluate:

- family probability fields,
- transition entropy fields,
- boundary overlap regions,
- OOD sweep entropy along controlled paths,
- soft assignment tables.

Expected prior outputs, when available:

```text
results/residual_universality_embedding.csv
results/ood_transfer_embedding.csv
results/ood_transfer_summary.csv
```

The notebook also includes fallback loaders for related outputs from Notebooks 18–23.

In [ ]:
# Notebook 24 setup

from pathlib import Path
import json
import math
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.spatial.distance import cdist
from scipy.special import logsumexp

SEED = 9423
rng = np.random.default_rng(SEED)

# Robust repo-root detection for Colab, local notebooks/, and GitHub clones.
CWD = Path.cwd()
if (CWD / "results").exists() or (CWD / "figures").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif CWD.name == "notebooks":
    REPO_ROOT = CWD.parent
elif (CWD.parent / "results").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
DOCS_DIR = REPO_ROOT / "docs"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("cwd:", CWD)
print("repo root:", REPO_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Load residual manifold and OOD sweep data

The loader looks for recent notebook outputs first. If an exact file is absent, it reconstructs a usable manifold from available residual feature files.

Files searched:

```text
results/residual_universality_embedding.csv
results/ood_transfer_embedding.csv
results/residual_pca_embedding.csv
results/residual_classification_feature_matrix.csv
results/residual_geometry_features.csv
results/ood_soft_universality_assignments.csv
```

In [ ]:
def read_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f"loaded: {p}")
            return pd.read_csv(p), p
    return None, None

known_candidates = [
    RESULTS_DIR / "residual_universality_embedding.csv",
    RESULTS_DIR / "residual_pca_embedding.csv",
    RESULTS_DIR / "residual_classification_feature_matrix.csv",
    RESULTS_DIR / "residual_geometry_features.csv",
]

ood_candidates = [
    RESULTS_DIR / "ood_transfer_embedding.csv",
    RESULTS_DIR / "ood_transfer_assignments.csv",
    RESULTS_DIR / "controlled_boundary_sweeps.csv",
    RESULTS_DIR / "ood_soft_universality_assignments.csv",
]

known_raw, known_path = read_first_existing(known_candidates)
ood_raw, ood_path = read_first_existing(ood_candidates)

available = sorted(p.name for p in RESULTS_DIR.glob("*"))
print("available result files:", available[:30], "..." if len(available) > 30 else "")

if known_raw is None:
    raise FileNotFoundError(
        "No known residual manifold file found. Run Notebook 18/20/21 first, "
        "or add one of: residual_universality_embedding.csv, residual_pca_embedding.csv, "
        "residual_classification_feature_matrix.csv, residual_geometry_features.csv."
    )

In [ ]:
# Normalize column names and identify topology / size columns.

def normalize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

known_raw = normalize_columns(known_raw)
if ood_raw is not None:
    ood_raw = normalize_columns(ood_raw)

def find_col(df, options):
    for c in options:
        if c in df.columns:
            return c
    lower = {c.lower(): c for c in df.columns}
    for c in options:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

top_col = find_col(known_raw, ["topology", "family", "known_family", "label"])
size_col = find_col(known_raw, ["N", "n", "graph_size", "n_modules", "size"])
pc1_col = find_col(known_raw, ["PC1", "pc1", "universality_PC1", "residual_manifold_coordinate_1"])
pc2_col = find_col(known_raw, ["PC2", "pc2", "universality_PC2", "residual_manifold_coordinate_2"])

print("known columns:", list(known_raw.columns))
print("topology column:", top_col)
print("size column:", size_col)
print("PC columns:", pc1_col, pc2_col)

In [ ]:
# Build known_df with topology, N, PC1, PC2.
# If PC1/PC2 are missing, compute PCA from numeric residual features.

known_df = known_raw.copy()

if top_col is None:
    raise ValueError("Known data needs a topology/family column.")

known_df["topology"] = known_df[top_col].astype(str)

if size_col is not None:
    known_df["N"] = pd.to_numeric(known_df[size_col], errors="coerce")
else:
    known_df["N"] = np.arange(len(known_df))

if pc1_col is not None and pc2_col is not None:
    known_df["PC1"] = pd.to_numeric(known_df[pc1_col], errors="coerce")
    known_df["PC2"] = pd.to_numeric(known_df[pc2_col], errors="coerce")
    embedding_source = f"loaded coordinates from {known_path.name}"
else:
    exclude = {top_col, size_col, "topology", "N"}
    numeric_cols = [
        c for c in known_df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(pd.to_numeric(known_df[c], errors="coerce"))
    ]
    if len(numeric_cols) < 2:
        raise ValueError("Need PC1/PC2 or at least two numeric feature columns for PCA.")
    Xnum = known_df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    Xnum = SimpleImputer(strategy="median").fit_transform(Xnum)
    Xnum = StandardScaler().fit_transform(Xnum)
    pca = PCA(n_components=2, random_state=SEED)
    coords = pca.fit_transform(Xnum)
    known_df["PC1"] = coords[:, 0]
    known_df["PC2"] = coords[:, 1]
    embedding_source = f"computed PCA from {len(numeric_cols)} numeric features in {known_path.name}"

known_df = known_df.dropna(subset=["PC1", "PC2", "topology"]).reset_index(drop=True)

# Clean labels for display.
label_map = {
    "erdos_renyi": "Erdős–Rényi",
    "erdos-renyi": "Erdős–Rényi",
    "erdos renyi": "Erdős–Rényi",
    "ring_lattice": "ring lattice",
    "small_world": "small world",
    "scale_free": "scale free",
    "modular_clustered": "modular clustered",
}
known_df["topology"] = known_df["topology"].replace(label_map)

print("embedding source:", embedding_source)
print(known_df[["topology", "N", "PC1", "PC2"]].head())
print("topologies:", sorted(known_df["topology"].unique()))

In [ ]:
# Build OOD dataframe when available. It can have PC1/PC2 already,
# or it can have numeric features compatible enough for a separate PCA fallback.
ood_df = None

if ood_raw is not None:
    ood_df = ood_raw.copy()
    ood_family_col = find_col(ood_df, ["ood_family", "sweep", "sweep_name", "family", "topology", "variant"])
    ood_param_col = find_col(ood_df, ["parameter", "param", "value", "sweep_value", "beta", "m", "p_out", "chord_density"])
    ood_size_col = find_col(ood_df, ["N", "n", "graph_size", "size"])
    ood_pc1_col = find_col(ood_df, ["PC1", "pc1", "universality_PC1", "residual_manifold_coordinate_1"])
    ood_pc2_col = find_col(ood_df, ["PC2", "pc2", "universality_PC2", "residual_manifold_coordinate_2"])

    if ood_family_col is None:
        ood_df["ood_family"] = "OOD sweep"
    else:
        ood_df["ood_family"] = ood_df[ood_family_col].astype(str)

    if ood_param_col is None:
        ood_df["parameter"] = np.arange(len(ood_df), dtype=float)
    else:
        ood_df["parameter"] = pd.to_numeric(ood_df[ood_param_col], errors="coerce")

    if ood_size_col is None:
        ood_df["N"] = np.nan
    else:
        ood_df["N"] = pd.to_numeric(ood_df[ood_size_col], errors="coerce")

    if ood_pc1_col is not None and ood_pc2_col is not None:
        ood_df["PC1"] = pd.to_numeric(ood_df[ood_pc1_col], errors="coerce")
        ood_df["PC2"] = pd.to_numeric(ood_df[ood_pc2_col], errors="coerce")
        ood_source_note = f"loaded OOD coordinates from {ood_path.name}"
    else:
        # Fallback: place OOD rows by nearest known feature centroids if no PC columns exist.
        # This keeps the notebook runnable, but direct PC columns are preferred.
        ood_df["PC1"] = np.nan
        ood_df["PC2"] = np.nan
        ood_source_note = f"OOD file found ({ood_path.name}) but no PC1/PC2 columns; OOD overlay skipped"

    ood_df = ood_df.dropna(subset=["PC1", "PC2"]).reset_index(drop=True)
    if len(ood_df) == 0:
        ood_df = None
        print(ood_source_note)
    else:
        print(ood_source_note)
        print(ood_df[["ood_family", "parameter", "N", "PC1", "PC2"]].head())
else:
    print("No OOD file found; density fields will still be produced for known topology manifold.")

## 2. Fit smooth topology density fields

Each known topology gets a Gaussian density in residual manifold coordinates.

For small sample counts, covariance regularization keeps density estimates stable.

In [ ]:
# Fit one regularized Gaussian density per topology.

topologies = sorted(known_df["topology"].unique())
coords = known_df[["PC1", "PC2"]].to_numpy(float)

# Grid bounds with padding.
x_min, x_max = np.nanmin(coords[:, 0]), np.nanmax(coords[:, 0])
y_min, y_max = np.nanmin(coords[:, 1]), np.nanmax(coords[:, 1])

if ood_df is not None and len(ood_df) > 0:
    ood_coords = ood_df[["PC1", "PC2"]].to_numpy(float)
    x_min = min(x_min, np.nanmin(ood_coords[:, 0]))
    x_max = max(x_max, np.nanmax(ood_coords[:, 0]))
    y_min = min(y_min, np.nanmin(ood_coords[:, 1]))
    y_max = max(y_max, np.nanmax(ood_coords[:, 1]))

x_pad = max(0.5, 0.15 * (x_max - x_min + 1e-9))
y_pad = max(0.5, 0.15 * (y_max - y_min + 1e-9))
x_grid = np.linspace(x_min - x_pad, x_max + x_pad, 180)
y_grid = np.linspace(y_min - y_pad, y_max + y_pad, 180)
XX, YY = np.meshgrid(x_grid, y_grid)
grid_points = np.column_stack([XX.ravel(), YY.ravel()])

def regularized_gaussian_logpdf(X, mu, cov):
    cov = np.asarray(cov, dtype=float)
    cov = cov + np.eye(2) * 1e-6
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0:
        cov = cov + np.eye(2) * 1e-3
        sign, logdet = np.linalg.slogdet(cov)
    inv = np.linalg.pinv(cov)
    delta = X - mu
    q = np.sum(delta @ inv * delta, axis=1)
    return -0.5 * (2 * np.log(2 * np.pi) + logdet + q)

density_models = {}
global_cov = np.cov(coords.T) + np.eye(2) * 0.05

for topo in topologies:
    sub = known_df[known_df["topology"] == topo][["PC1", "PC2"]].to_numpy(float)
    mu = sub.mean(axis=0)
    if len(sub) >= 3:
        cov = np.cov(sub.T)
    elif len(sub) == 2:
        cov = np.cov(sub.T) + 0.25 * global_cov
    else:
        cov = 0.5 * global_cov
    # Shared regularization smooths sparse topologies.
    cov = 0.7 * cov + 0.3 * global_cov + np.eye(2) * 0.08
    density_models[topo] = {"mu": mu, "cov": cov, "n": len(sub)}

logps = []
for topo in topologies:
    model = density_models[topo]
    prior = np.log(model["n"] / len(known_df))
    logps.append(regularized_gaussian_logpdf(grid_points, model["mu"], model["cov"]) + prior)
logps = np.vstack(logps).T

lse = logsumexp(logps, axis=1, keepdims=True)
probs = np.exp(logps - lse)

entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1) / np.log(len(topologies))
top_idx = np.argmax(probs, axis=1)
top_prob = probs[np.arange(len(probs)), top_idx]
sorted_probs = np.sort(probs, axis=1)
top_two_gap = sorted_probs[:, -1] - sorted_probs[:, -2]

grid_df = pd.DataFrame({
    "PC1": grid_points[:, 0],
    "PC2": grid_points[:, 1],
    "entropy": entropy,
    "top_family": [topologies[i] for i in top_idx],
    "top_probability": top_prob,
    "top_two_gap": top_two_gap,
})
for i, topo in enumerate(topologies):
    grid_df[f"p_{topo}"] = probs[:, i]

grid_path = RESULTS_DIR / "continuous_density_grid.csv"
grid_df.to_csv(grid_path, index=False)
print("saved:", grid_path)
grid_df.head()

## 3. Soft universality density map

The background shows the highest-probability topology at each residual-manifold location.

Known family centroids are marked by black stars. Known samples are transparent circles. OOD trajectories are dashed with x markers when OOD data is available.

In [ ]:
# Plot soft universality density map.

topo_to_int = {t: i for i, t in enumerate(topologies)}
Z_family = np.array([topo_to_int[t] for t in grid_df["top_family"]]).reshape(XX.shape)
Z_conf = grid_df["top_probability"].to_numpy().reshape(XX.shape)

fig, ax = plt.subplots(figsize=(12, 8))

# Family map background.
im = ax.imshow(
    Z_family,
    extent=[x_grid.min(), x_grid.max(), y_grid.min(), y_grid.max()],
    origin="lower",
    aspect="auto",
    alpha=0.20,
)

# Confidence contour.
cs = ax.contour(XX, YY, Z_conf, levels=[0.45, 0.60, 0.75, 0.90], linewidths=1, alpha=0.55)
ax.clabel(cs, inline=True, fontsize=8)

# Known samples and centroids.
for topo in topologies:
    sub = known_df[known_df["topology"] == topo]
    ax.scatter(sub["PC1"], sub["PC2"], s=90, alpha=0.45, label=topo)
    mu = density_models[topo]["mu"]
    ax.scatter(mu[0], mu[1], marker="*", s=380, c="black", edgecolors="white", linewidths=0.8)
    ax.text(mu[0] + 0.05, mu[1] + 0.05, topo, fontsize=11, weight="bold")

# OOD overlays.
if ood_df is not None and len(ood_df) > 0:
    for fam, sub in ood_df.sort_values("parameter").groupby("ood_family"):
        ax.plot(sub["PC1"], sub["PC2"], "--", marker="x", linewidth=2, alpha=0.8, label=f"OOD: {fam}")

ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_title("Soft universality density map")
ax.set_xlabel("residual manifold coordinate 1")
ax.set_ylabel("residual manifold coordinate 2")
ax.legend(loc="best", fontsize=9)
fig.tight_layout()

fig_path = FIGURES_DIR / "24_soft_universality_density_map.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

## 4. Transition entropy field

High entropy indicates overlap between topology density fields. These regions are candidate transition corridors or ambiguous boundary zones.

In [ ]:
Z_entropy = grid_df["entropy"].to_numpy().reshape(XX.shape)

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.contourf(XX, YY, Z_entropy, levels=24, alpha=0.85)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("normalized transition entropy")

for topo in topologies:
    mu = density_models[topo]["mu"]
    ax.scatter(mu[0], mu[1], marker="*", s=350, c="black")
    ax.text(mu[0] + 0.05, mu[1] + 0.05, topo, fontsize=11, weight="bold")

if ood_df is not None and len(ood_df) > 0:
    for fam, sub in ood_df.sort_values("parameter").groupby("ood_family"):
        ax.plot(sub["PC1"], sub["PC2"], "--", marker="x", linewidth=2, alpha=0.9, label=fam)
    ax.legend(loc="best", fontsize=8)

ax.axhline(0, color="white", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvline(0, color="white", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_title("Transition entropy field")
ax.set_xlabel("residual manifold coordinate 1")
ax.set_ylabel("residual manifold coordinate 2")
fig.tight_layout()

fig_path = FIGURES_DIR / "24_transition_entropy_field.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

## 5. Boundary overlap field and boundary thickness summary

Boundary overlap is identified where the top-two probability gap is small. This approximates transition thickness in residual-manifold coordinates.

In [ ]:
# Boundary overlap: top two family probabilities close.
gap_threshold = 0.15
boundary_mask = grid_df["top_two_gap"].to_numpy() < gap_threshold
Z_boundary = boundary_mask.reshape(XX.shape).astype(float)
Z_gap = grid_df["top_two_gap"].to_numpy().reshape(XX.shape)

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.contourf(XX, YY, 1 - Z_gap, levels=24, alpha=0.85)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("overlap score = 1 - top-two probability gap")

ax.contour(XX, YY, Z_boundary, levels=[0.5], colors="black", linewidths=1.2)

for topo in topologies:
    mu = density_models[topo]["mu"]
    ax.scatter(mu[0], mu[1], marker="*", s=350, c="black")
    ax.text(mu[0] + 0.05, mu[1] + 0.05, topo, fontsize=11, weight="bold")

if ood_df is not None and len(ood_df) > 0:
    for fam, sub in ood_df.sort_values("parameter").groupby("ood_family"):
        ax.plot(sub["PC1"], sub["PC2"], "--", marker="x", linewidth=2, alpha=0.9)

ax.set_title("Boundary overlap field")
ax.set_xlabel("residual manifold coordinate 1")
ax.set_ylabel("residual manifold coordinate 2")
fig.tight_layout()

fig_path = FIGURES_DIR / "24_boundary_overlap_field.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

# Summary per top family.
cell_area = (x_grid[1] - x_grid[0]) * (y_grid[1] - y_grid[0])
boundary_summary = (
    grid_df.assign(is_boundary=boundary_mask)
    .groupby("top_family")
    .agg(
        grid_cells=("PC1", "size"),
        boundary_cells=("is_boundary", "sum"),
        mean_entropy=("entropy", "mean"),
        mean_top_probability=("top_probability", "mean"),
        mean_top_two_gap=("top_two_gap", "mean"),
    )
    .reset_index()
)
boundary_summary["boundary_fraction"] = boundary_summary["boundary_cells"] / boundary_summary["grid_cells"]
boundary_summary["boundary_area_estimate"] = boundary_summary["boundary_cells"] * cell_area

boundary_path = RESULTS_DIR / "boundary_thickness_summary.csv"
boundary_summary.to_csv(boundary_path, index=False)
print("saved:", boundary_path)
boundary_summary

## 6. Soft assignment for known and OOD points

Each point receives:

- top probability family,
- top probability value,
- entropy,
- top-two gap,
- full family probability vector.

In [ ]:
def soft_assign_points(df, family_col=None):
    X = df[["PC1", "PC2"]].to_numpy(float)
    point_logps = []
    for topo in topologies:
        model = density_models[topo]
        prior = np.log(model["n"] / len(known_df))
        point_logps.append(regularized_gaussian_logpdf(X, model["mu"], model["cov"]) + prior)
    point_logps = np.vstack(point_logps).T
    point_lse = logsumexp(point_logps, axis=1, keepdims=True)
    point_probs = np.exp(point_logps - point_lse)

    idx = np.argmax(point_probs, axis=1)
    sorted_p = np.sort(point_probs, axis=1)
    ent = -np.sum(point_probs * np.log(point_probs + 1e-12), axis=1) / np.log(len(topologies))

    out = df.copy()
    out["soft_family"] = [topologies[i] for i in idx]
    out["soft_probability"] = point_probs[np.arange(len(point_probs)), idx]
    out["soft_entropy"] = ent
    out["top_two_gap"] = sorted_p[:, -1] - sorted_p[:, -2]
    for i, topo in enumerate(topologies):
        out[f"p_{topo}"] = point_probs[:, i]
    return out

known_soft = soft_assign_points(known_df)
known_soft["source"] = "known"

if ood_df is not None and len(ood_df) > 0:
    ood_soft = soft_assign_points(ood_df)
    ood_soft["source"] = "OOD"
else:
    ood_soft = pd.DataFrame()

known_soft_path = RESULTS_DIR / "known_soft_universality_assignments.csv"
known_soft.to_csv(known_soft_path, index=False)
print("saved:", known_soft_path)

if len(ood_soft) > 0:
    ood_soft_path = RESULTS_DIR / "ood_soft_universality_assignments.csv"
    ood_soft.to_csv(ood_soft_path, index=False)
    print("saved:", ood_soft_path)
    display_cols = ["ood_family", "parameter", "N", "soft_family", "soft_probability", "soft_entropy", "top_two_gap"]
    display(ood_soft[display_cols].head(12))
else:
    print("OOD soft assignments skipped because no OOD coordinates were available.")

## 7. OOD entropy along sweeps

This plot tracks whether controlled sweep paths pass through high-entropy ambiguity zones.

In [ ]:
if len(ood_soft) > 0:
    families = sorted(ood_soft["ood_family"].unique())
    fig, ax = plt.subplots(figsize=(12, 7))
    for fam in families:
        sub = ood_soft[ood_soft["ood_family"] == fam].sort_values("parameter")
        ax.plot(sub["parameter"], sub["soft_entropy"], marker="o", linewidth=2, label=fam)
    ax.set_title("OOD trajectory entropy along controlled sweeps")
    ax.set_xlabel("sweep parameter")
    ax.set_ylabel("soft assignment entropy")
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.35)
    fig.tight_layout()
    fig_path = FIGURES_DIR / "24_ood_entropy_along_sweeps.png"
    fig.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", fig_path)
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.text(0.5, 0.5, "OOD coordinates unavailable", ha="center", va="center", fontsize=14)
    ax.set_axis_off()
    fig_path = FIGURES_DIR / "24_ood_entropy_along_sweeps.png"
    fig.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved placeholder:", fig_path)

In [ ]:
if len(ood_soft) > 0:
    summary = (
        ood_soft.groupby("ood_family")
        .agg(
            n_points=("soft_entropy", "size"),
            mean_entropy=("soft_entropy", "mean"),
            max_entropy=("soft_entropy", "max"),
            min_top_two_gap=("top_two_gap", "min"),
            mean_soft_probability=("soft_probability", "mean"),
            n_family_switches=("soft_family", lambda s: int((s != s.shift()).sum() - 1) if len(s) > 1 else 0),
        )
        .reset_index()
        .sort_values(["max_entropy", "n_family_switches"], ascending=False)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    s1 = summary.sort_values("max_entropy")
    axes[0].barh(s1["ood_family"], s1["max_entropy"])
    axes[0].set_title("maximum OOD entropy")
    axes[0].set_xlabel("max entropy")

    s2 = summary.sort_values("mean_soft_probability")
    axes[1].barh(s2["ood_family"], s2["mean_soft_probability"])
    axes[1].set_title("mean soft assignment confidence")
    axes[1].set_xlabel("mean top probability")

    fig.suptitle("Soft assignment confidence across OOD sweeps", y=1.02)
    fig.tight_layout()
    fig_path = FIGURES_DIR / "24_soft_assignment_confidence.png"
    fig.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", fig_path)

    ood_summary_path = RESULTS_DIR / "ood_soft_assignment_summary.csv"
    summary.to_csv(ood_summary_path, index=False)
    print("saved:", ood_summary_path)
    display(summary)
else:
    summary = pd.DataFrame()

## 8. Export notebook summary

This JSON is intended for README/paper integration.

In [ ]:
summary_payload = {
    "notebook": "24_continuous_universality_density_fields",
    "seed": SEED,
    "embedding_source": embedding_source,
    "known_path": str(known_path) if known_path else None,
    "ood_path": str(ood_path) if ood_path else None,
    "n_known_points": int(len(known_df)),
    "n_ood_points": int(len(ood_soft)) if len(ood_soft) > 0 else 0,
    "topologies": topologies,
    "grid_cells": int(len(grid_df)),
    "mean_grid_entropy": float(grid_df["entropy"].mean()),
    "max_grid_entropy": float(grid_df["entropy"].max()),
    "boundary_gap_threshold": gap_threshold,
    "boundary_fraction_total": float(boundary_mask.mean()),
    "exports": {
        "continuous_density_grid": str(RESULTS_DIR / "continuous_density_grid.csv"),
        "known_soft_assignments": str(RESULTS_DIR / "known_soft_universality_assignments.csv"),
        "ood_soft_assignments": str(RESULTS_DIR / "ood_soft_universality_assignments.csv") if len(ood_soft) > 0 else None,
        "boundary_thickness_summary": str(RESULTS_DIR / "boundary_thickness_summary.csv"),
        "figures": [
            str(FIGURES_DIR / "24_soft_universality_density_map.png"),
            str(FIGURES_DIR / "24_transition_entropy_field.png"),
            str(FIGURES_DIR / "24_boundary_overlap_field.png"),
            str(FIGURES_DIR / "24_ood_entropy_along_sweeps.png"),
            str(FIGURES_DIR / "24_soft_assignment_confidence.png"),
        ],
    },
}

summary_path = RESULTS_DIR / "notebook_24_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2))
print(json.dumps(summary_payload, indent=2))
print("saved:", summary_path)

## 9. Optional zip download for Colab

Run this cell in Colab when you want one zip containing Notebook 24 outputs.

In [ ]:
# Optional Colab download cell.
# Uncomment files.download(...) when running in Google Colab.

import zipfile

zip_path = REPO_ROOT / "notebook_24_outputs.zip"
wanted = [
    RESULTS_DIR / "continuous_density_grid.csv",
    RESULTS_DIR / "known_soft_universality_assignments.csv",
    RESULTS_DIR / "ood_soft_universality_assignments.csv",
    RESULTS_DIR / "ood_soft_assignment_summary.csv",
    RESULTS_DIR / "boundary_thickness_summary.csv",
    RESULTS_DIR / "notebook_24_summary.json",
    FIGURES_DIR / "24_soft_universality_density_map.png",
    FIGURES_DIR / "24_transition_entropy_field.png",
    FIGURES_DIR / "24_boundary_overlap_field.png",
    FIGURES_DIR / "24_ood_entropy_along_sweeps.png",
    FIGURES_DIR / "24_soft_assignment_confidence.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in wanted:
        if p.exists():
            zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print("created:", zip_path)

# In Colab:
# from google.colab import files
# files.download(str(zip_path))